In [ ]:
!pip install wilds matplotlib torch torchvision terratorch


In [ ]:
from wilds import get_dataset
import pandas.core.tools.datetimes as _pd_dt_module
import matplotlib.pyplot as plt
import pandas as pd


my_root_path = "/mnt/windows/Users/kevin/Downloads/"

_true_orig_to_datetime = _pd_dt_module.to_datetime

def _patched_to_datetime(*args, **kwargs):
    # Always force ISO8601 — overrides missing OR incorrect format
    kwargs['format'] = 'ISO8601'
    return _true_orig_to_datetime(*args, **kwargs)

pd.to_datetime = _patched_to_datetime

#never use dataset, but use the metadata(it contains all IMAGES)
dataset = get_dataset(dataset="fmow", root_dir=my_root_path, download=False)

# Restore
pd.to_datetime = _true_orig_to_datetime

# Grab the pre-built, rich pandas DataFrame directly from the dataset
df = dataset.metadata.copy()

# Extract the timestamp safely by explicitly defining the ISO8601 format
df['timestamp'] = pd.to_datetime(df['timestamp'], format='ISO8601')
df['year_extracted'] = df['timestamp'].dt.year
#Find the exact column index for 'region' in the metadata_array
region_idx = dataset.metadata_fields.index('region')

# Retrieve the mapping list from the dataset (e.g., ['Asia', 'Europe', ...])
region_names_list = dataset.metadata_map['region']

df['region_names'] = df['region'].map(lambda x: region_names_list[int(x)])

In [ ]:
# ---------------------------------------------------------
#  Simplified DataLoader
# ---------------------------------------------------------
import torch
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD)
])
def get_input(idx):
        """
        Returns x for a given idx.
        """
        img = Image.open(Path(my_root_path + 'fmow_v1.1') / 'images' / f'rgb_img_{idx}.png').convert('RGB')
        return img


def get_tensor_images(numberOfImages):
    # Standard ImageNet statistics (commonly used for RGB satellite imagery too)
    lista_tensori = []
    for i in range(0,numberOfImages):
        img = get_input(i)
        img_t = transform(img)
        lista_tensori.append(img_t)

    batch_immagini = torch.stack(lista_tensori,dim = 0)
    return batch_immagini



In [ ]:
import FoundationModel
# %%
# ---------------------------------------------------------
# CLUSTERING DEGLI EMBEDDINGS (Creazione dei Concepts)
# ---------------------------------------------------------
import torch
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from tqdm import tqdm
import gc

print("--- Inizio Estrazione Embeddings e Clustering ---")

# 1. Campionamento di 1000 immagini random dal dataframe
# Usiamo random_state per la riproducibilità
df_sample = df.sample(n=2000, random_state=42).copy()
sample_indices = df_sample.index.tolist()


# Carichiamo il modello
fm_model = FoundationModel.FoundationModel()



In [ ]:
# 3. Estrazione degli Embeddings (Inference Loop)
batch_size = 128
all_embeddings = []

with torch.no_grad():
    for i in tqdm(range(0, len(sample_indices), batch_size), desc="Estrazione feature FM"):
        # Prendi gli indici per il batch corrente
        batch_idx = sample_indices[i:i+batch_size]

        batch_imgs = []
        for idx in batch_idx:
            # Utilizziamo le tue funzioni già definite: get_input e transform
            img = get_input(idx)
            img_t = transform(img)
            batch_imgs.append(img_t)

        # Stack
        input_tensor = torch.stack(batch_imgs, dim=0)

        # Forward pass nel Foundation Model
        features = fm_model(input_tensor)

        # Sposta su CPU, converti in numpy array e salva
        all_embeddings.append(features.cpu().numpy())

        # Cleanup memoria (fondamentale con dataset grandi)
        del input_tensor
        del features
        gc.collect()

# Concateniamo tutti i batch in un'unica matrice (number_of_samples, dim_embedding)
embeddings_matrix = np.concatenate(all_embeddings, axis=0)
print(f"Forma della matrice degli embeddings: {embeddings_matrix.shape}")

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans

print("\n--- Avvio Clustering basato sulle Classi (Task/Class Shift) ---")

# 1. Uniamo gli embeddings temporaneamente con le etichette di classe
df_emb = pd.DataFrame(embeddings_matrix)
df_emb['category'] = df_sample['category'].values

# 2. Calcoliamo il Centroide per ogni singola classe (media matematica dei vettori)
class_centroids = df_emb.groupby('category').mean()

nomi_classi = class_centroids.index.tolist()
matrice_centroidi = class_centroids.values

print(f"Calcolati {len(nomi_classi)} centroidi di classe nello spazio a 192 dimensioni.")

# 3. Eseguiamo il K-Means SUI CENTROIDI, non sulle immagini!
num_concepts = 1
print(f"Raggruppamento delle {len(nomi_classi)} classi in {num_concepts} Concepts esclusivi...")

kmeans_classi = KMeans(n_clusters=num_concepts, random_state=41, n_init='auto')
labels_concept = kmeans_classi.fit_predict(matrice_centroidi)

# 4. Creiamo un dizionario di mappatura { "Nome_Classe": ID_Concept }
mappatura_classi = dict(zip(nomi_classi, labels_concept))

df_sample['concept_cluster'] = df_sample['category'].map(mappatura_classi)

# --- Verifica del risultato ---
print("\n--- Classi assegnate a ciascun Concept ---")
for cluster_id in range(num_concepts):
    classi_nel_concept = [cls for cls, c_id in mappatura_classi.items() if c_id == cluster_id]


print("\n--- Numero di Immagini per ogni Concept ---")
# Calcola quante immagini ci sono in ogni cluster in un colpo solo
conteggi_immagini = df_sample['concept_cluster'].value_counts()

for cluster_id in range(num_concepts):
    numero_immagini = conteggi_immagini.get(cluster_id, 0)

    print(f"[ Concept {cluster_id} ]: {numero_immagini} immagini")

In [ ]:
import torch
from torch import nn
from terratorch import BACKBONE_REGISTRY

class StreamingTerraModel(nn.Module):
    def __init__(self, num_classes=62):
        super().__init__()
        self.backbone = BACKBONE_REGISTRY.build(
            "terramind_v1_base",
            pretrained=True,
            modalities=["RGB"]
        )
        print(self.backbone)
        # Freeze the backbone
        for param in self.backbone.parameters():
            param.requires_grad = False

        self.backbone.eval()

        self.head = nn.Linear(in_features=768, out_features=num_classes)

    def forward(self, inputTensor):
        # 1. Extract features safely
        with torch.no_grad():
            features = self.backbone(inputTensor)[-1]
            # We flatten the spatial dimensions and average them out.
            # This safely converts the tensor into a clean [Batch, 192] vector.
        features = features.mean(dim=1)

        logits = self.head(features)

        return logits

In [ ]:
import torch
from torch import nn
from tqdm import tqdm
import Student

# 1. PREPARAZIONE DELLO STREAM (Ordinamento per Concept)
# Ordiniamo il dataset in base al concept_cluster: prima tutto il Concept 0, poi l'1, ecc.
df_stream = df_sample.sort_values(by='concept_cluster').copy()

# 2. MAPPATURA DELLE CLASSI (Da Stringa a Intero 0-61)
# CrossEntropyLoss richiede etichette numeriche. Creiamo un dizionario per convertirle.
#possiamo utilizzare il mapping che c'è nel dataset ma per ora va bene cosi
classi_uniche = sorted(df_stream['category'].unique())
class_to_idx = {nome_classe: indice for indice, nome_classe in enumerate(classi_uniche)}

# BASIC TRIAL WITH ONLY STUDENT
student = StreamingTerraModel()

In [ ]:
optimizer = torch.optim.Adam(student.head.parameters(), lr=1e-3)
ce_loss_fn = nn.CrossEntropyLoss()

batch_size = 32

buffer_imgs = []
buffer_labels = []

# Variabili per tracciare le performance
concept_corrente = -1
concept_corrette = 0
concept_viste = 0

print("--- Avvio Streaming Pipeline (Prequential Evaluation) ---")

# Iteriamo sulle righe del DataFrame ordinato
for index, row in tqdm(df_stream.iterrows(), total=len(df_stream), desc="Stream"):

    # Se il concept cambia, stampiamo un avviso (Simulazione Concept Drift!)
    if row['concept_cluster'] != concept_corrente:
        if concept_corrente != -1 and concept_viste > 0:
            acc_finale = (concept_corrette / concept_viste) * 100
            print(f"\n---> [FINE CONCEPT {concept_corrente}] Accuratezza Chiusura: {acc_finale:.2f}% <---")
        # Rilevato nuovo Concept: Reset dei contatori!
        print(f"\n[!] CONCEPT DRIFT: Inizio Concept {row['concept_cluster']}")
        concept_corrente = row['concept_cluster']
        concept_corrette = 0
        concept_viste = 0
    # --- 1. CARICAMENTO DATI REALI ---
    # Usiamo il vero indice dell'immagine per caricarla tramite la tua funzione
    img_pil = get_input(index)
    img_tensor = transform(img_pil).unsqueeze(0)

    # Convertiamo l'etichetta stringa nel numero intero corrispondente
    label_idx = class_to_idx[row['category']]
    label_tensor = torch.tensor([label_idx], dtype=torch.long)

    # --- 2. PREQUENTIAL EVALUATION (TEST) ---
    student.eval()
    with torch.no_grad():

        prediction_logits = student(img_tensor)
        predicted_class = torch.argmax(prediction_logits, dim=1)

        # Aggiorniamo le metriche del Concept Corrente
        if predicted_class.item() == label_idx:
            concept_corrette += 1
        concept_viste += 1

    # --- 3. AGGIUNTA AL BUFFER ---
    buffer_imgs.append(img_tensor)
    buffer_labels.append(label_tensor)

    # --- 4. ONLINE LEARNING (TRAIN SUL BATCH) ---
    if len(buffer_imgs) == batch_size:
        student.train() # Modalità allenamento
        batch_imgs = torch.cat(buffer_imgs, dim=0)
        batch_labels = torch.cat(buffer_labels, dim=0)
        for epoch in range(0,1):
            optimizer.zero_grad()
            # Forward pass sull'intero batch
            student_output = student(batch_imgs)
            logits = student_output
            # Calcolo Loss e Backward
            loss_ce = ce_loss_fn(logits, batch_labels)
            loss_ce.backward()
            optimizer.step()

        # Stampa dell'accuratezza cumulata fino a questo punto del Concept
        acc_corrente = (concept_corrette / concept_viste) * 100
        print(f"Concept {concept_corrente} | Sample visti: {concept_viste} | Classi corrette : {concept_corrette} | Accuratezza Cumulata: {acc_corrente:.2f}%")

            # Svuotiamo i buffer per il prossimo ciclo
        buffer_imgs.clear()
        buffer_labels.clear()

In [ ]:

# ---------------------------------------------------------
# 5. LINEAR PROBING OFFLINE CON FIX (UPPER BOUND INTRA-CONCEPT)
# ---------------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from collections import Counter

print("\n--- Avvio Linear Probing OFFLINE (Scikit-Learn) ---")

concept_target = 0

# Filtriamo gli embeddings e le etichette appartenenti SOLO al Concept 0
mask = (df_sample['concept_cluster'] == concept_target).values
X_concept = embeddings_matrix[mask]
y_concept = df_sample['category'].values[mask]

# --- FIX PER LE CLASSI RARE E PYARROW ---
conteggio_classi = Counter(y_concept)
# Teniamo solo le classi che hanno almeno 2 campioni
classi_valide = [cls for cls, count in conteggio_classi.items() if count >= 2]

# Creiamo una nuova maschera per tenere solo i dati delle classi valide
maschera_valide = np.isin(y_concept, classi_valide)
X_concept_filtered = X_concept[maschera_valide]

# Forziamo il tipo a stringa NumPy standard per evitare l'errore ArrowExtensionArray
y_concept_filtered = np.array(y_concept[maschera_valide]).astype(str)

print(f"Campioni iniziali per il Concept {concept_target}: {len(X_concept)}")
print(f"Campioni dopo rimozione classi rare: {len(X_concept_filtered)}")

# Dividiamo in Train (80%) e Test (20%) usando i dati filtrati
X_train, X_test, y_train, y_test = train_test_split(
    X_concept_filtered, y_concept_filtered, test_size=0.2, random_state=42, stratify=y_concept_filtered
)

# Inizializziamo il classificatore lineare
linear_head_offline = LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1)

print(f"Addestramento del Linear Head offline (Concept {concept_target})...")
linear_head_offline.fit(X_train, y_train)

train_preds = linear_head_offline.predict(X_train)
test_preds = linear_head_offline.predict(X_test)

acc_train = accuracy_score(y_train, train_preds) * 100
acc_test = accuracy_score(y_test, test_preds) * 100

print(f"Accuratezza in Addestramento: {acc_train:.2f}%")
print(f"---> UPPER BOUND REALE (Test Set Concept {concept_target}): {acc_test:.2f}% <---")